# Create the fact_sales table
###Each row of this fact table shows the product purchased in a unique order.
### The dimensions of this fact table are the **Product**, the **Reseller** (who made the purchase), the **Employee** (who handled the sale), the **SalesTerritory** (where the purchase was made) and the **Date** of the order. The measures include the **quantity** of the product purchased in the order, the **unit price** (the price at which the product was sold) the **sales amount** (unit price * quantity) and the **cost** of the products sold (product cost * quantity).

Read sales table from silver schema and dim_date table

In [0]:
sales = table("sales_lakehouse.silver.sales")

In [0]:
display(sales.limit(10))

In [0]:
print(f"Number of rows in sales table {sales.count()}")

In [0]:
dim_date = spark.table("sales_lakehouse.gold.dim_date")
display(dim_date.limit(5))

**We create the fact_sales table by joining with dim_date to obtain the date_key foreign key, replacing the order_date column. We also generate a unique fact_sales_key.**

In [0]:
from pyspark.sql.functions import col, to_date, monotonically_increasing_id

staging_fact_sales = (
    # Join sales and dim_date tables
    sales.alias("s")
    .join(                                                  
        dim_date.alias("d"),
        col("s.order_date") == col("d.date"), 
        "left"
    )
    # Select the columns for the fact_sales table
    .select(
        col("s.sales_order_number"),
        col("d.date_key"),
        col("s.product_key"),
        col("s.reseller_key"),
        col("s.employee_key"),
        col("s.sales_territory_key"),
        col("s.quantity"),
        col("s.sales"),
        col("s.cost")
    )
    # Create the unique fact_sales_key
    .withColumn("fact_sales_key", monotonically_increasing_id()) 
)

# Reorder columns
staging_fact_sales = staging_fact_sales.select(
    "fact_sales_key",
    "sales_order_number",
    "date_key",
    "product_key",
    "reseller_key",
    "employee_key",
    "sales_territory_key",
    "quantity",
    "sales",
    "cost"

)

display(staging_fact_sales.limit(10))

In [0]:
# Row count check to ensure no records were lost when creating the fact table
print(f"Number of rows in sales table {staging_fact_sales.count()}")

In [0]:
staging_fact_sales.write.format("delta").mode("overwrite").saveAsTable("sales_lakehouse.gold.fact_sales")